# 🚀 Kaggle Worker Notebook - Ollama (Qwen 2.5 7B) & Ngrok Static Tunnel
Notebook này chạy Ollama GPU trên Kaggle và kết nối trực tiếp với Static Domain Ngrok về máy local.

In [ ]:
# [1] Cài đặt zstd (Bắt buộc để giải nén Ollama) & Ollama Linux
print("🚀 [1/3] Đang cài đặt zstd và Ollama Linux...")
!apt-get update -y > /dev/null 2>&1
!apt-get install zstd -y > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install --quiet pyngrok httpx
print("✅ Cài đặt Ollama thành công!")

In [ ]:
# [2] Cấu hình đĩa lưu trữ & Khởi chạy Ollama Server ngầm
import os, time, subprocess, httpx

os.environ["OLLAMA_MODELS"] = "/kaggle/working/ollama_models"
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
os.makedirs("/kaggle/working/ollama_models", exist_ok=True)

print("⚡ [2/3] Khởi động Ollama Server ngầm...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Kiểm tra Ollama đã sẵn sàng chưa
for _ in range(10):
    try:
        res = httpx.get("http://127.0.0.1:11434/api/tags", timeout=3.0)
        if res.status_code == 200:
            print("✅ Ollama Local Server đang hoạt động tốt!")
            break
    except Exception:
        time.sleep(2)

print("📥 Đang tải mô hình Qwen 2.5 7B GGUF (vui lòng chờ trong 1-2 phút)... ")
!ollama pull qwen2.5:7b
print("✅ Tải model Qwen 2.5 7B hoàn tất!")

In [ ]:
# [3] Kết nối Static Ngrok Tunnel & Duy trì kết nối liên tục
import time, httpx
from pyngrok import ngrok

# 1. Điền Authtoken ngrok
NGROK_AUTHTOKEN = "3Hl420hPJiQrcgDIWpked30M1Ca_7rfpvf1wR4MzLUK4pmMPw"
STATIC_DOMAIN = "bondless-immerse-paternal.ngrok-free.dev"

ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill() # Đóng các tunnel cũ nếu có

try:
    print(f"🚀 Đang kết nối ngrok tới domain cố định: {STATIC_DOMAIN} ...")
    tunnel = ngrok.connect(11434, domain=STATIC_DOMAIN)
    print(f"🎉 KẾT NỐI THÀNH CÔNG! URL: {tunnel.public_url}")
    print("🔥 FastAPI Local của bạn có thể gọi thẳng tới URL này!")
    print("⏳ Giữ cell này chạy liên tục để không bị ngắt kết nối...")
    
    # Giữ loop để Kaggle không ngắt session
    count = 0
    while True:
        time.sleep(30)
        count += 30
        # Heartbeat check
        try:
            r = httpx.get("http://127.0.0.1:11434/api/tags", timeout=5.0)
            print(f"[Heartbeat - {count}s] Ollama & Ngrok đang chạy ổn định! Model count: {len(r.json().get('models', []))}")
        except Exception as err:
            print(f"⚠️ Cảnh báo: {err}")
except Exception as e:
    print(f"❌ Lỗi khi khởi tạo Ngrok: {e}")